# Quinta Playa — beach change pipeline

Built from scratch on everything learned so far. Every number this notebook reports comes
with the uncertainty attached.

## What the Metashape reports show (read this first)

Each month was processed with **different alignment and camera-calibration parameters**:

| | Dec2024 | Jan2025 | Feb2025 | Mar2025 |
|---|---|---|---|---|
| Camera loc. error | 3.03 cm | 1.64 cm | **1.88 m** | 20.68 cm |
| Calib fitted | k1-k4 | +b1,b2, k1-k4 | k1-k3 | k1-k3 |
| Keypoint/Mpx | 40,000 | 40,000 | 1,000 | 1,000 |
| Markers | 33, **set 1** | 10, set 2 | 10, set 2 | 10, set 2 |
| Altitude | **63.7 m** | 44.2 m | 45.0 m | 44.0 m |

Residual lens distortion not absorbed by the calibration model is the classic cause of
systematic bowl/dome deformation in SfM. Different models per month means **different
deformation per month** — which is what the stable-feature disagreement has been measuring.

**No post-hoc alignment fixes this**, because it isn't a rigid error. Reprocessing all
months with identical parameters is the real fix. This notebook quantifies where things
stand today and produces defensible numbers with stated limits.

## Pipeline

| Phase | Does | Output |
|---|---|---|
| 0 | Config, GCPs, shapefile | — |
| 1 | Measure roughness, choose M3C2 radii | `normal_radii`, `cyl_radius` |
| 2 | Clip all clouds to the shapefile | clipped LAZ |
| 3 | Align on stable boxes (rigid, multi-box) | one transform per month |
| 4 | Validate: LOO + GCP holdout + residual maps | `registration_error` |
| 5 | Decision gate | go / marginal / stop |
| 6 | M3C2 | distances + LoD |
| 7 | Maps, masked by detectability | figures |
| 8 | Summary table | CSV |


## Phase 0 — Configuration

In [ ]:
from pathlib import Path
import gc, json, itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from pyproj import Transformer

import py4dgeo

OUTDIR = Path("results"); OUTDIR.mkdir(exist_ok=True)
CLIPDIR = Path("clipped"); CLIPDIR.mkdir(exist_ok=True)

# --- Months in chronological order. Order matters for every table and figure. ---
MONTHS = {
    "Dec2024": "E:/Finalised/2025/December/PointCloud_Dec_2025_set2.laz",
    "Jan2025": "E:/Finalised/2025/January/PointCloud_Jan_2025.laz",
    "Feb2025": "E:/Finalised/2025/February/PointCloud_Feb_2025.laz",
    "Mar2025": "E:/Finalised/2025/March/PointCloud_Mar_2025.laz",
    "Apr2025": "E:/Finalised/2025/April/PointCloud_Apr_2025.laz",
    "May2025": "E:/Finalised/2025/May/PointCloud_May_2025.laz",
}
REFERENCE = "Jan2025"     # best camera location error (1.64 cm) of the reports seen

# --- Clip boundary. One shapefile, applied to every month. ---
SHAPEFILE = "E:/Finalised/2025/February/Shapefile_Feb_2025.laz"   # EDIT
CLIP_BUFFER = 5.0   # m. M3C2 needs neighbours beyond each corepoint; without a
                    # buffer, edge corepoints get truncated cylinders and biased
                    # normals. Must exceed cyl_radius + max(normal_radii).

# --- Stable boxes: rigid ground used to align months to each other. ---
STABLE_BOXES = {
    "rock1":   ((714334.860, 9888847.230), (714344.000, 9888851.060)),
    "rock2":   ((712940.840, 9888569.620), (712941.670, 9888571.150)),
    "rock3":   ((713035.910, 9888585.540), (713036.330, 9888586.940)),
    "rock4":   ((714619.920, 9888768.340), (714623.930, 9888770.580)),
    "rock5":   ((714334.335, 9888867.658), (714335.103, 9888868.571)),
    "rooftop": ((713015.340, 9888681.810), (713023.640, 9888691.370)),
}

# --- The 6 trusted GCPs. NOT used for alignment -- they are the holdout test. ---
GCP_WGS84 = {
    "A1_HITOCONTROL_QP": (-91.08768304, -1.00784055, 3.87360000),
    "B1_HITOCONTROL_QP": (-91.08546709, -1.00675111, 3.01130000),
    "B2_HITOCONTROL_QP": (-91.08228286, -1.00562935, 3.25120000),
    "B3_HITOCONTROL_QP2": (-91.07993045, -1.00516543, 3.38860000),
    "C1_HITOCONTROL_QP4": (-91.07321937, -1.00486026, 3.76150000),
    "B4_HITOCONTROL_QP7": (-91.07477345, -1.00459609, 4.00260000),
}

_tf = Transformer.from_crs("EPSG:4326", "EPSG:32715", always_xy=True)
_names = list(GCP_WGS84)
_lon = np.array([GCP_WGS84[n][0] for n in _names])
_lat = np.array([GCP_WGS84[n][1] for n in _names])
_alt = np.array([GCP_WGS84[n][2] for n in _names])
_e, _n = _tf.transform(_lon, _lat)
GCP = pd.DataFrame({"name": _names, "x": _e, "y": _n, "z": _alt})

assert 7.1e5 < GCP.x.mean() < 7.2e5, "eastings not in UTM 15S range for this site"
print(GCP.to_string(index=False))

# --- Decision thresholds, fixed before seeing any result. ---
PASS_CM, FAIL_CM = 5.0, 15.0

# --- Corepoint spacing for M3C2. Radii are MEASURED in Phase 1, not set here. ---
COREPOINT_SPACING = 1.0
MAX_DISTANCE = 5.0
ROBUST_AGGR = True


## Phase 1 — Measure roughness, then choose the M3C2 radii

Lague et al. (2013): the normal-estimation **diameter** should be at least 20–25x the
surface roughness, so the **radius** should be at least 10–12x it. Roughness here means the
scatter of points about a locally fitted plane — which depends on the scale you measure it
at, so it gets measured at several scales and the smallest radius satisfying the criterion
is chosen.

Measured on the **sand**, not the rocks: sand is what M3C2 has to work on, and it is the
harder surface (low texture, so noisier photogrammetric reconstruction).


In [ ]:
def local_roughness(cloud, scales, n_sample=1500, max_nb=2000, seed=0):
    """Median std of residuals about a locally fitted plane, per scale."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(cloud), min(n_sample, len(cloud)), replace=False)
    tree = cKDTree(cloud)
    out = {}
    for r in scales:
        vals = []
        for i in idx:
            nb = tree.query_ball_point(cloud[i], r)
            if len(nb) < 10:
                continue
            if len(nb) > max_nb:
                nb = rng.choice(nb, max_nb, replace=False)
            P = cloud[nb]
            P = P - P.mean(axis=0)
            # smallest singular vector = plane normal; residual spread along it
            normal = np.linalg.svd(P, full_matrices=False)[2][2]
            vals.append(np.std(P @ normal))
        out[r] = float(np.median(vals)) if vals else np.nan
    return out


SCALES = [0.10, 0.25, 0.50, 1.00, 2.00, 4.00]

print(f"sampling {REFERENCE} ...")
_ep = py4dgeo.read_from_las(MONTHS[REFERENCE])
_rng = np.random.default_rng(0)
_sub = _ep.cloud[_rng.choice(len(_ep.cloud), min(3_000_000, len(_ep.cloud)), replace=False)]
del _ep; gc.collect()

rough = local_roughness(_sub, SCALES)
print(f"\n{'radius m':>10}{'roughness m':>14}{'10x roughness':>16}{'radius OK?':>12}")
for r, v in rough.items():
    print(f"{r:>10.2f}{v:>14.4f}{10*v:>16.2f}{'yes' if r >= 10*v else 'no':>12}")

ok = [r for r, v in rough.items() if np.isfinite(v) and r >= 10 * v]
NORMAL_RADII = [ok[0], ok[0] * 2, ok[0] * 4] if ok else [0.5, 1.0, 2.0]
CYL_RADIUS = NORMAL_RADII[0]

print(f"\nNORMAL_RADII = {NORMAL_RADII}   (multiscale; py4dgeo picks the most planar per corepoint)")
print(f"CYL_RADIUS   = {CYL_RADIUS}")
print("\nIf nothing satisfied the criterion the defaults are used -- check the table above.")
assert CLIP_BUFFER > CYL_RADIUS + max(NORMAL_RADII), "CLIP_BUFFER too small for these radii"
del _sub; gc.collect()


## Phase 2 — Clip every cloud to the shapefile

Removes the tide problem: ground outside the boundary is not comparable between months.
Written once to `clipped/`, so everything downstream is smaller and faster.

`CLIP_BUFFER` keeps a margin beyond the boundary so M3C2 has full neighbourhoods at the
edge of the analysis area.


In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
from shapely import contains_xy
import laspy

_g = gpd.read_file(SHAPEFILE)
if _g.crs is not None and _g.crs.to_epsg() != 32715:
    _g = _g.to_crs(32715)
BOUNDARY = unary_union(_g.geometry.values)
BOUNDARY_BUF = BOUNDARY.buffer(CLIP_BUFFER)
print(f"boundary area {BOUNDARY.area:,.0f} m2   buffered {BOUNDARY_BUF.area:,.0f} m2")


def clip_laz(src, dst, poly, chunk=10_000_000):
    """Stream through the file so a 400M-point cloud never loads whole."""
    kept = total = 0
    with laspy.open(src) as f:
        hdr = f.header
        with laspy.open(dst, mode="w", header=hdr) as w:
            for pts in f.chunk_iterator(chunk):
                total += len(pts)
                m = contains_xy(poly, np.asarray(pts.x), np.asarray(pts.y))
                if m.any():
                    w.append_points(pts[m])
                    kept += int(m.sum())
    return kept, total


CLIPPED = {}
for name, src in MONTHS.items():
    dst = CLIPDIR / f"{name}_clipped.laz"
    CLIPPED[name] = str(dst)
    if dst.exists():
        print(f"{name:<10}already clipped, skipping")
        continue
    kept, total = clip_laz(src, str(dst), BOUNDARY_BUF)
    print(f"{name:<10}{kept:>14,} / {total:>14,} kept ({100*kept/total:.1f}%)")


## Phase 3 — Align on the stable boxes

Same structure as py4dgeo's registration tutorial — crop, `iterative_closest_point`, read
the 4x4 affine matrix — extended to several boxes at once.

Two details that matter:

**Equal weighting.** The rooftop has ~160,000 points and rock3 has ~1,400. Unweighted, ICP
would be driven almost entirely by the rooftop. Each box is subsampled to the same count
before they are combined.

**Reduction point.** The matrix must be applied as `R(x - x0) + t + x0`. Raw UTM coordinates
are ~7e5, so rotation math without recentring loses precision. `Epoch.transform()` handles
this; never multiply the matrix by hand.


In [ ]:
def crop_to_bbox(cloud, bbox):
    (xmin, ymin), (xmax, ymax) = bbox
    mask = (
        (cloud[:, 0] >= xmin) & (cloud[:, 0] <= xmax) &
        (cloud[:, 1] >= ymin) & (cloud[:, 1] <= ymax)
    )
    cropped = cloud[mask]
    if cropped.shape[0] == 0:
        raise ValueError(f"Bounding box {bbox} matched 0 points -- check coordinates and CRS")
    return cropped


def stable_crops(cloud, boxes):
    return {k: crop_to_bbox(cloud, b) for k, b in boxes.items()}


def balanced_union(crops, seed=0, cap=None):
    """Combine boxes with equal weight: subsample each to the smallest count."""
    rng = np.random.default_rng(seed)
    n = min(len(v) for v in crops.values())
    if cap is not None:
        n = min(n, cap)
    return np.vstack([v[rng.choice(len(v), n, replace=False)] for v in crops.values()]), n


# Load the stable crops for every month once, then free the clouds.
crops_by_month = {}
for name, path in CLIPPED.items():
    ep = py4dgeo.read_from_las(path)
    crops_by_month[name] = stable_crops(ep.cloud, STABLE_BOXES)
    sizes = "  ".join(f"{k}:{len(v):,}" for k, v in crops_by_month[name].items())
    print(f"{name:<10}{sizes}")
    del ep; gc.collect()


In [ ]:
def align(ref_crops, mov_crops, boxes=None, seed=0):
    """Rigid ICP of `mov` onto `ref` using the named boxes. Returns the py4dgeo
    Transformation plus the balanced clouds used, so residuals can be recomputed."""
    boxes = boxes or list(ref_crops)
    r_union, n = balanced_union({k: ref_crops[k] for k in boxes}, seed)
    m_union, _ = balanced_union({k: mov_crops[k] for k in boxes}, seed)

    ref_ep = py4dgeo.Epoch(r_union)
    mov_ep = py4dgeo.Epoch(m_union)
    red_poi = r_union.mean(axis=0)   # keeps rotation math near the origin

    trafo = py4dgeo.iterative_closest_point(
        ref_ep, mov_ep, tolerance=0.00001, max_iterations=50, reduction_point=red_poi
    )
    return trafo, r_union, m_union, n


def apply_trafo(cloud, trafo):
    """Always via py4dgeo -- it applies the reduction point correctly."""
    ep = py4dgeo.Epoch(cloud.copy())
    ep.transform(transformation=trafo)
    return ep.cloud


def box_offset(ref_crop, mov_crop, trafo=None):
    """Median XYZ offset at one box, optionally after applying a transform.
    Nearest-neighbour in XY then difference -- robust for sparse crops where a
    full 6-DOF ICP would be poorly constrained."""
    mov = apply_trafo(mov_crop, trafo) if trafo is not None else mov_crop
    d, idx = cKDTree(ref_crop[:, :2]).query(mov[:, :2], distance_upper_bound=0.05)
    ok = np.isfinite(d)
    if ok.sum() < 5:
        return np.array([np.nan] * 3), 0
    diff = ref_crop[idx[ok]] - mov[ok]
    return np.median(diff, axis=0), int(ok.sum())


TRAFOS = {}
print(f"{'month':<10}{'dx cm':>9}{'dy cm':>9}{'dz cm':>9}{'pts/box':>10}")
for m in MONTHS:
    if m == REFERENCE:
        continue
    trafo, r_u, m_u, n = align(crops_by_month[REFERENCE], crops_by_month[m])
    TRAFOS[m] = trafo
    dx, dy, dz = trafo.affine_transformation[:3, 3]
    print(f"{m:<10}{dx*100:>9.2f}{dy*100:>9.2f}{dz*100:>9.2f}{n:>10,}")

print(f"\nExample affine matrix ({list(TRAFOS)[0]}):")
print(TRAFOS[list(TRAFOS)[0]].affine_transformation)


## Phase 4 — Validation

Three independent checks. They answer different questions and can disagree.

### 4a. Leave-one-out

A rigid transform has 6 parameters and there are 6 stable boxes, so an in-sample residual is
close to guaranteed to look good and proves very little. Instead: fit on 5 boxes, measure the
error at the 6th, repeat 6 times. **The LOO RMS is the honest registration uncertainty** and
is what gets passed to M3C2 as `registration_error`.


In [ ]:
loo_rows = []
for m in MONTHS:
    if m == REFERENCE:
        continue
    for held in STABLE_BOXES:
        fit_on = [b for b in STABLE_BOXES if b != held]
        trafo, *_ = align(crops_by_month[REFERENCE], crops_by_month[m], boxes=fit_on)
        off, npts = box_offset(crops_by_month[REFERENCE][held],
                               crops_by_month[m][held], trafo)
        loo_rows.append(dict(month=m, held_out=held,
                             dx=off[0], dy=off[1], dz=off[2],
                             horiz=np.hypot(off[0], off[1]), n=npts))

loo = pd.DataFrame(loo_rows)
print(loo.assign(**{c: (loo[c] * 100).round(1) for c in ["dx", "dy", "dz", "horiz"]})
         .to_string(index=False))

LOO_RMS = {}
print(f"\n{'month':<10}{'LOO horiz RMS cm':>18}{'LOO vert RMS cm':>18}{'LOO 3D RMS cm':>16}")
for m, g in loo.groupby("month", sort=False):
    h = np.sqrt(np.nanmean(g.horiz ** 2))
    v = np.sqrt(np.nanmean(g.dz ** 2))
    t = np.sqrt(np.nanmean(g.dx ** 2 + g.dy ** 2 + g.dz ** 2))
    LOO_RMS[m] = dict(horiz=h, vert=v, total=t)
    print(f"{m:<10}{h*100:>18.1f}{v*100:>18.1f}{t*100:>16.1f}")


### 4b. GCP holdout

The 6 GCPs were **not used in the alignment**, so checking them afterwards is a genuine
independent test against known truth.

Locating a painted hito in a point cloud needs both position and colour. Colour alone fails
(other red things exist on a beach); position alone fails on flat ground, where a blob of
points gives elevation but no horizontal signal. So: search a small radius around the known
coordinate, then take the reddest points **within that neighbourhood only**. Their centroid
is the detected marker.


In [ ]:
GCP_SEARCH_RADIUS = 1.5   # m around the surveyed coordinate
GCP_RED_PCT = 15          # keep the reddest N% inside that radius


def load_near_gcps(path, gcp_xy, radius, chunk=10_000_000):
    """Stream the file, keep only points near a GCP, with RGB."""
    xyz_keep, rgb_keep = [], []
    with laspy.open(path) as f:
        dims = {d.name for d in f.header.point_format.dimensions}
        has_rgb = {"red", "green", "blue"} <= dims
        for pts in f.chunk_iterator(chunk):
            x, y = np.asarray(pts.x), np.asarray(pts.y)
            m = np.zeros(len(x), dtype=bool)
            for gx, gy in gcp_xy:
                m |= ((x - gx) ** 2 + (y - gy) ** 2) < radius ** 2
            if not m.any():
                continue
            xyz_keep.append(np.column_stack([x[m], y[m], np.asarray(pts.z)[m]]))
            if has_rgb:
                rgb_keep.append(np.column_stack([np.asarray(pts.red)[m],
                                                 np.asarray(pts.green)[m],
                                                 np.asarray(pts.blue)[m]]).astype(float))
    if not xyz_keep:
        return np.empty((0, 3)), None
    return np.vstack(xyz_keep), (np.vstack(rgb_keep) if rgb_keep else None)


def detect_marker(xyz, rgb, gcp_xy, radius, red_pct):
    """Centroid of the reddest points within `radius` of the surveyed position."""
    near = ((xyz[:, 0] - gcp_xy[0]) ** 2 + (xyz[:, 1] - gcp_xy[1]) ** 2) < radius ** 2
    if near.sum() < 20:
        return None, 0
    P = xyz[near]
    if rgb is None:
        return P.mean(axis=0), int(near.sum())   # position only, no colour available
    col = rgb[near]
    redness = col[:, 0] - 0.5 * (col[:, 1] + col[:, 2])
    sel = redness >= np.percentile(redness, 100 - red_pct)
    if sel.sum() < 5:
        return None, int(near.sum())
    return P[sel].mean(axis=0), int(sel.sum())


gcp_xy = GCP[["x", "y"]].to_numpy()
gcp_rows = []
for m, path in CLIPPED.items():
    xyz, rgb = load_near_gcps(path, gcp_xy, GCP_SEARCH_RADIUS)
    if m != REFERENCE:
        xyz = apply_trafo(xyz, TRAFOS[m])     # check AFTER alignment
    for i, row in GCP.iterrows():
        pos, n = detect_marker(xyz, rgb, (row.x, row.y), GCP_SEARCH_RADIUS, GCP_RED_PCT)
        if pos is None:
            gcp_rows.append(dict(month=m, gcp=row["name"], dx=np.nan, dy=np.nan,
                                 dz=np.nan, n=n))
            continue
        gcp_rows.append(dict(month=m, gcp=row["name"],
                             dx=pos[0] - row.x, dy=pos[1] - row.y, dz=pos[2] - row.z, n=n))
    del xyz, rgb; gc.collect()

gcps = pd.DataFrame(gcp_rows)
gcps["horiz"] = np.hypot(gcps.dx, gcps.dy)
print(gcps.assign(**{c: (gcps[c] * 100).round(1) for c in ["dx", "dy", "dz", "horiz"]})
          .to_string(index=False))

print(f"\n{'month':<10}{'GCP horiz RMS cm':>18}{'GCP vert RMS cm':>18}{'n found':>10}")
GCP_RMS = {}
for m, g in gcps.groupby("month", sort=False):
    h = np.sqrt(np.nanmean(g.horiz ** 2)); v = np.sqrt(np.nanmean(g.dz ** 2))
    GCP_RMS[m] = dict(horiz=h, vert=v)
    print(f"{m:<10}{h*100:>18.1f}{v*100:>18.1f}{int(g.dz.notna().sum()):>10}")

print("\nThese are an INDEPENDENT check -- the GCPs never entered the alignment.")
print("If GCP RMS is much worse than LOO RMS, the stable boxes are not representative")
print("of the whole site and the LOO number is optimistic.")


### 4c. Residual structure

A small RMS can still hide structure. Coherent patterns mean the rigid model is wrong — which
is exactly what a per-month lens-model difference would produce, since that deformation is not
rigid and no rigid transform can remove it.


In [ ]:
months_cmp = [m for m in MONTHS if m != REFERENCE]
fig, axes = plt.subplots(1, len(months_cmp), figsize=(3.6 * len(months_cmp), 3.8), squeeze=False)

for ax, m in zip(axes[0], months_cmp):
    g = gcps[gcps.month == m]
    sc = ax.scatter(GCP.x, GCP.y, c=g.dz.values * 100, cmap="RdBu_r",
                    s=90, vmin=-20, vmax=20, edgecolor="k", linewidth=0.4)
    for _, row in loo[loo.month == m].iterrows():
        b = STABLE_BOXES[row.held_out]
        cx, cy = (b[0][0] + b[1][0]) / 2, (b[0][1] + b[1][1]) / 2
        ax.scatter([cx], [cy], c=[row.dz * 100], cmap="RdBu_r",
                   s=55, vmin=-20, vmax=20, marker="s", edgecolor="k", linewidth=0.4)
    ax.set_title(f"{m}\ncircles=GCP, squares=LOO box", fontsize=8)
    ax.set_aspect("equal"); ax.ticklabel_format(style="plain", useOffset=False)
    ax.tick_params(labelsize=6)
    plt.colorbar(sc, ax=ax, label="dz (cm)")

plt.tight_layout(); plt.savefig(OUTDIR / "residual_structure.png", dpi=150); plt.show()
print("A smooth gradient across either symbol type = non-rigid deformation a rigid")
print("transform cannot remove. Scattered signs with no pattern = random noise.")


## Phase 5 — Decision gate

Thresholds were fixed in Phase 0, before any result was seen.

| Vertical LOO RMS | Verdict |
|---|---|
| under 5 cm | Vertical change is defensible above ~2x that. |
| 5–15 cm | Marginal. Large features only, floor stated everywhere. |
| over 15 cm | Vertical not usable. Horizontal metrics only. |

Horizontal is judged separately and has consistently been far better.


In [ ]:
worst_v = max(r["vert"] for r in LOO_RMS.values()) * 100
worst_h = max(r["horiz"] for r in LOO_RMS.values()) * 100

print(f"worst vertical LOO RMS   {worst_v:.1f} cm")
print(f"worst horizontal LOO RMS {worst_h:.1f} cm\n")

if worst_v < PASS_CM:
    VERDICT = "GO"
    print(f"GO. Vertical change above ~{1.96*worst_v:.0f} cm is defensible.")
elif worst_v < FAIL_CM:
    VERDICT = "MARGINAL"
    print(f"MARGINAL. Quote {1.96*worst_v:.0f} cm as the vertical detection floor.")
    print("Report large features only. Do not add polynomial corrections -- with 6")
    print("stable boxes you would be fitting noise.")
else:
    VERDICT = "STOP"
    print("STOP for vertical. A rigid alignment does not bring these months onto a")
    print("common vertical frame -- consistent with the per-month lens models in the")
    print("Metashape reports. Report horizontal metrics, and reprocess with identical")
    print("parameters before revisiting vertical change.")

print(f"\nHorizontal remains usable at ~{1.96*worst_h:.0f} cm regardless.")


## Phase 6 — M3C2

`registration_error` is set **per month** from that month's LOO vertical RMS, so
`lodetection` reflects real uncertainty rather than reporting millimetre sensitivity the
data cannot support.

Corepoints are voxel-subsampled for even ground spacing. Index slicing (`cloud[::50]`) follows
file order, not position, so coverage ends up uneven.


In [ ]:
py4dgeo.enable_trace(False); py4dgeo.enable_timeit(False)

ref_ep = py4dgeo.read_from_las(CLIPPED[REFERENCE])
vapc = py4dgeo.Vapc(ref_ep, voxel_size=COREPOINT_SPACING)
corepoints = vapc.reduce_to_feature("closest_to_centroids").epoch.cloud
# keep corepoints inside the boundary; the buffer exists only to feed M3C2 neighbourhoods
corepoints = corepoints[contains_xy(BOUNDARY, corepoints[:, 0], corepoints[:, 1])]
print(f"corepoints: {corepoints.shape[0]:,} at {COREPOINT_SPACING} m spacing")
np.save(OUTDIR / "corepoints.npy", corepoints)
del vapc, ref_ep; gc.collect()


In [ ]:
import collections

DIST, UNC, LOD = {}, {}, {}

for m in months_cmp:
    print(f"\n=== {REFERENCE} -> {m} ===")
    ep_ref = py4dgeo.read_from_las(CLIPPED[REFERENCE])
    ep_mov = py4dgeo.read_from_las(CLIPPED[m])
    ep_mov.transform(transformation=TRAFOS[m])      # alignment from Phase 3

    m3c2 = py4dgeo.M3C2(
        epochs=(ep_ref, ep_mov), corepoints=corepoints,
        normal_radii=NORMAL_RADII, cyl_radius=CYL_RADIUS,
        max_distance=MAX_DISTANCE,
        registration_error=LOO_RMS[m]["vert"],      # per month, from Phase 4a
        robust_aggr=ROBUST_AGGR,
    )
    d, u = m3c2.run()
    DIST[m] = d
    UNC[m] = {k: np.asarray(u[k]) for k in
              ("lodetection", "spread1", "spread2", "num_samples1", "num_samples2")}
    LOD[m] = float(np.nanmedian(UNC[m]["lodetection"]))
    radii = collections.Counter(np.asarray(m3c2.directions_radii()).ravel())

    fin = np.isfinite(d)
    print(f"  {fin.sum():,}/{len(d):,} corepoints returned a distance")
    print(f"  median change      {np.nanmedian(d[fin])*100:+.1f} cm")
    print(f"  roughness spread1  {np.nanmedian(UNC[m]['spread1'])*100:.1f} cm")
    print(f"  pts per cylinder   {np.nanmedian(UNC[m]['num_samples1']):.0f}")
    print(f"  LoD (median)       {LOD[m]*100:.1f} cm")
    print(f"  normal radius used {dict(radii)}")

    np.save(OUTDIR / f"m3c2_{REFERENCE}_{m}.npy", d)
    np.save(OUTDIR / f"lod_{REFERENCE}_{m}.npy", UNC[m]["lodetection"])
    del ep_ref, ep_mov, m3c2; gc.collect()


## Phase 7 — Maps

Two rows, chronological, one shared colour scale so panels are comparable.

Top: everything. Bottom: only corepoints exceeding their own `lodetection`. The bottom row
is the honest figure — plotting sub-LoD values is how a registration artefact becomes
published erosion.


In [ ]:
lim = np.nanpercentile(np.abs(np.concatenate([DIST[m][np.isfinite(DIST[m])] for m in months_cmp])), 95)

fig, axes = plt.subplots(2, len(months_cmp), figsize=(3.7 * len(months_cmp), 8), squeeze=False)
for j, m in enumerate(months_cmp):
    d = DIST[m]; fin = np.isfinite(d)
    sig = fin & (np.abs(d) > UNC[m]["lodetection"])

    ax = axes[0][j]
    sc = ax.scatter(corepoints[fin, 0], corepoints[fin, 1], c=d[fin],
                    cmap="RdBu_r", vmin=-lim, vmax=lim, s=1.5)
    ax.set_title(f"{REFERENCE} -> {m}\nall values", fontsize=8)

    ax = axes[1][j]
    ax.scatter(corepoints[fin, 0], corepoints[fin, 1], c="0.9", s=1)
    if sig.sum():
        ax.scatter(corepoints[sig, 0], corepoints[sig, 1], c=d[sig],
                   cmap="RdBu_r", vmin=-lim, vmax=lim, s=1.5)
    ax.set_title(f"above LoD ({LOD[m]*100:.0f} cm)\n{100*sig.sum()/max(fin.sum(),1):.0f}% resolvable",
                 fontsize=8)

for ax in axes.ravel():
    ax.set_aspect("equal"); ax.ticklabel_format(style="plain", useOffset=False)
    ax.tick_params(labelsize=6)
fig.colorbar(sc, ax=axes, label="M3C2 distance (m)", shrink=0.6)
plt.savefig(OUTDIR / "change_maps.png", dpi=150, bbox_inches="tight"); plt.show()


## Phase 8 — Summary

Every number needed to state results with their uncertainty, in one table.


In [ ]:
rows = []
for m in months_cmp:
    d = DIST[m]; fin = np.isfinite(d)
    sig = fin & (np.abs(d) > UNC[m]["lodetection"])
    rows.append(dict(
        month=m,
        loo_horiz_cm=round(LOO_RMS[m]["horiz"] * 100, 1),
        loo_vert_cm=round(LOO_RMS[m]["vert"] * 100, 1),
        gcp_horiz_cm=round(GCP_RMS[m]["horiz"] * 100, 1),
        gcp_vert_cm=round(GCP_RMS[m]["vert"] * 100, 1),
        roughness_cm=round(float(np.nanmedian(UNC[m]["spread1"])) * 100, 1),
        lod_cm=round(LOD[m] * 100, 1),
        median_change_cm=round(float(np.nanmedian(d[fin])) * 100, 1),
        pct_resolvable=round(100 * sig.sum() / max(fin.sum(), 1), 1),
    ))

summary = pd.DataFrame(rows)
summary.to_csv(OUTDIR / "summary.csv", index=False)
print(summary.to_string(index=False))

print(f"""
HOW TO STATE THIS
-----------------
Alignment: rigid ICP on {len(STABLE_BOXES)} stable features, validated leave-one-out.
Uncertainty: quote loo_vert_cm per month -- not the in-sample residual.
Independent check: gcp_*_cm, from 6 GCPs held out of the alignment entirely.
Detection limit: lod_cm. Values below it are not measurements.
Verdict: {VERDICT} for vertical change. Horizontal is usable throughout.

CAVEAT THAT OUTRANKS ALL OF THE ABOVE
The Metashape reports show each month was processed with different alignment and camera
calibration parameters (Jan fitted b1,b2,k1-k4; Feb/Mar only k1-k3; camera location error
ranges 1.64 cm to 1.88 m). Residual lens distortion produces systematic surface
deformation, and a different lens model per month produces different deformation per
month. That is not a rigid error and no alignment in this notebook can remove it.
Reprocessing all months with identical parameters is the real fix.
""")
